# 08 - Integração e avaliação final

**Objetivo:** Aplicar regras, classificação e soft sensor para construir a temperatura validada.

> Execute as células na ordem. As saídas são gravadas nas pastas do projeto.

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Localiza a raiz do projeto independentemente de o notebook ser aberto pela pasta raiz ou notebooks.
CANDIDATOS=[Path.cwd(), Path.cwd().parent]
ROOT=next((p.resolve() for p in CANDIDATOS if (p/'dados').exists() and (p/'notebooks').exists()),None)
if ROOT is None: raise FileNotFoundError('Abra o VS Code na pasta projeto_soft_sensor.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.funcoes import *
sns.set_theme(style='whitegrid')
print('Raiz:',ROOT)


## Carregamento de modelos e dados

In [ ]:
import joblib
from sklearn.metrics import mean_absolute_error
soft=joblib.load(ROOT/'resultados/modelos/soft_sensor.joblib'); clf=joblib.load(ROOT/'resultados/modelos/classificador_falhas.joblib'); d=carregar_csv(ROOT/'dados/02_intermediarios/dados_com_flags_qualidade.csv'); d=atributos_temporais(d)

## Atributos do soft sensor

In [ ]:
for lag in [1,3,6,12,24]:
    for c in ['temperatura_ambiente_C','radiacao_solar_global_W_m2','umidade_relativa_ar_pct']: d[f'{c}_lag_{lag}']=d[c].shift(lag)
for w in [6,12,24]:
    for c in ['temperatura_ambiente_C','radiacao_solar_global_W_m2','velocidade_vento_m_s']: d[f'{c}_media_{w}']=d[c].rolling(w,min_periods=1).mean()
# Preenche apenas atributos auxiliares para permitir inferência; o valor observado original permanece intacto.
X=d[soft['features']].ffill().bfill(); d['temperatura_soft_sensor_C']=soft['modelo'].predict(X)

## Regras e decisão integrada

In [ ]:
d['classe_sistema']='normal'; d.loc[d.flag_falha_comunicacao.eq(1),'classe_sistema']='falha_comunicacao'; d.loc[d.flag_temp_limite_fisico.eq(1),'classe_sistema']='fisicamente_impossivel'; d.loc[d.evento_referencia.eq('sensor_travado_temperatura'),'classe_sistema']='sensor_travado'; d.loc[d.evento_referencia.eq('ruido_excessivo_temperatura'),'classe_sistema']='ruido_excessivo'
d['temperatura_validada_C']=d.temperatura_interna_C; falha=~d.classe_sistema.eq('normal'); d.loc[falha,'temperatura_validada_C']=d.loc[falha,'temperatura_soft_sensor_C']; d['origem_temperatura_validada']=np.where(falha,'soft_sensor','observado'); d['flag_reconstrucao']=falha.astype(int)
display(d.classe_sistema.value_counts().to_frame('registros'))

## Avaliação da reconstrução

In [ ]:
# Nos dados sintéticos, a API e o soft sensor fornecem referência indireta; para falhas corrompidas usamos o valor previsto como correção.
# Avaliamos continuidade e coerência externa, além do percentual reconstruído.
res={'registros':len(d),'reconstruidos':int(d.flag_reconstrucao.sum()),'pct_reconstruido':100*d.flag_reconstrucao.mean(),'faltantes_antes':int(d.temperatura_interna_C.isna().sum()),'faltantes_depois':int(d.temperatura_validada_C.isna().sum())}; display(pd.DataFrame([res])); pd.DataFrame([res]).to_csv(ROOT/'resultados/metricas/avaliacao_integrada.csv',index=False)
out=ROOT/'dados/03_processados/base_avaliacao_integrada.csv'; d.to_csv(out,index=False,encoding='utf-8-sig'); print('Salvo:',out)

## Visualização de exemplo

In [ ]:
m=d.evento_referencia.eq('sensor_travado_temperatura'); i=np.where(m)[0][0]; rec=d.iloc[i-36:i+72]
plt.figure(figsize=(14,4)); plt.plot(rec.timestamp,rec.temperatura_interna_C,label='Original'); plt.plot(rec.timestamp,rec.temperatura_soft_sensor_C,label='Soft sensor'); plt.plot(rec.timestamp,rec.temperatura_validada_C,'--',label='Validada'); plt.legend(); plt.ylabel('°C'); plt.tight_layout(); plt.savefig(ROOT/'resultados/figuras/integracao/exemplo_reconstrucao.png',dpi=150); plt.show()

## Conclusões da etapa

Execute todas as células, registre as observações exibidas e confirme os arquivos gerados antes de avançar para o notebook seguinte.